---

## **DIPLOME UNIVERSITAIRE SDA**


## **Projet Generative AI — Système Agentique d'Évaluation et d'Anticipation des Risques Climatiques**





Promotion 007

Avril 2026




**Corpus** : 10 rapports scientifiques (GIEC AR6, Copernicus, EM-DAT, NOAA, JRC, WMO, EU CELEX)

**Repo** : https://github.com/diegomerchanm/catastrophes-climatiques-rag

---

**Ce notebook est conçu pour être :**
- **reproductible** (chemins relatifs, seeds fixées)
- **idempotent** (relançable sans recalculer si les fichiers existent déjà)
- **traçable** (quality gates go/no-go explicites)

**Convention :** chaque cellule de code doit produire une sortie visible. Si aucune sortie naturelle, ajouter un `print()` de vérification.

---

# NOTEBOOK 7 — MCP (Model Context Protocol) & Email

**Auteur :** Xia Bizot

---

### Objectif

Démontrer que les outils du projet sont exposés via le protocole MCP et accessibles depuis Claude Desktop. Inclut le test de l'envoi d'alertes par email.

---

### Plan du notebook

| Section | Contenu |
|---|---|
| 1. Configuration | Imports, chemins, vérifications |
| 2. Présentation MCP | Qu'est-ce que c'est, architecture |
| 3. Test des outils via appel direct | 7 outils testés |
| 4. Test de l'email | Envoi d'alerte |
| 5. Serveur MCP | Analyse du code, configuration Claude Desktop |
| 6. Résultats | Tableaux |
| 7. Conclusions | Quality gate |

---

### Hypothèse testable

> Les outils exposés via MCP produisent des résultats identiques aux appels directs, permettant l'intégration dans Claude Desktop sans modification du code source.

---

---

## 1. Configuration

In [ ]:
import os
import sys
import time
import logging
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

SEED = 42
notebook_start_time = time.time()

BASE = Path('..')
OUTPUT_DIR = BASE / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(BASE))

from dotenv import load_dotenv
load_dotenv(BASE / '.env')

print('>> 1. Configuration : OK')

---

## 2. Présentation MCP

**MCP (Model Context Protocol)** est un protocole qui expose des outils pour qu'ils soient utilisables par n'importe quel client compatible (Claude Desktop, etc.).

Notre serveur MCP (`mcp_server.py`) expose 7 outils :

| Outil MCP | Outil interne | Fonction |
|---|---|---|
| `meteo_actuelle` | `get_weather` | Météo temps réel |
| `meteo_historique` | `get_historical_weather` | Météo passée |
| `previsions_meteo` | `get_forecast` | Prévisions 7 jours |
| `recherche_web` | `web_search` | Tavily + DuckDuckGo |
| `calculatrice` | `calculator` | Calculs math |
| `recherche_corpus` | `search_corpus` | RAG hybride BM25+Dense |
| `envoyer_email` | `send_email` | Alertes climatiques |

---

---

## 3. Test des outils via appel direct

On teste chaque outil directement (sans passer par MCP) pour valider qu'ils fonctionnent.

In [ ]:
from src.agents.tools import (
    get_weather, get_historical_weather, get_forecast,
    web_search, calculator, search_corpus, send_email,
)

results_outils = []

# Test 1 : Météo actuelle
t0 = time.time()
r = get_weather.invoke({'city': 'Paris'})
d = round(time.time() - t0, 2)
print(f'get_weather(Paris) : {r[:100]}...')
results_outils.append({'outil': 'get_weather', 'input': 'Paris', 'duree_s': d, 'statut': '[OK]' if 'Paris' in r else '[KO]'})

# Test 2 : Météo historique
t0 = time.time()
r = get_historical_weather.invoke({'city': 'Marseille', 'date': '2023-10-15'})
d = round(time.time() - t0, 2)
print(f'get_historical_weather(Marseille, 2023-10-15) : {r[:100]}...')
results_outils.append({'outil': 'get_historical_weather', 'input': 'Marseille 2023-10-15', 'duree_s': d, 'statut': '[OK]' if 'Marseille' in r else '[KO]'})

# Test 3 : Prévisions
t0 = time.time()
r = get_forecast.invoke({'city': 'Lyon'})
d = round(time.time() - t0, 2)
print(f'get_forecast(Lyon) : {r[:100]}...')
results_outils.append({'outil': 'get_forecast', 'input': 'Lyon', 'duree_s': d, 'statut': '[OK]' if 'Lyon' in r else '[KO]'})

# Test 4 : Recherche web
t0 = time.time()
r = web_search.invoke({'query': 'catastrophes climatiques 2024', 'max_results': 3})
d = round(time.time() - t0, 2)
print(f'web_search(catastrophes 2024) : {r[:100]}...')
results_outils.append({'outil': 'web_search', 'input': 'catastrophes 2024', 'duree_s': d, 'statut': '[OK]' if 'Résultats' in r or 'résultats' in r.lower() else '[KO]'})

# Test 5 : Calculatrice
t0 = time.time()
r = calculator.invoke({'expression': '3+7*2'})
d = round(time.time() - t0, 2)
print(f'calculator(3+7*2) : {r}')
results_outils.append({'outil': 'calculator', 'input': '3+7*2', 'duree_s': d, 'statut': '[OK]' if '17' in r else '[KO]'})

# Test 6 : Email (sans config = message d'erreur attendu)
t0 = time.time()
r = send_email.invoke({'destinataire': 'test@test.com', 'sujet': 'Test NB7', 'contenu': 'Test'})
d = round(time.time() - t0, 2)
print(f'send_email(test) : {r[:100]}')
results_outils.append({'outil': 'send_email', 'input': 'test@test.com', 'duree_s': d, 'statut': '[OK]'})

print('\n>> 3. Test outils direct : OK')

---

## 4. Serveur MCP — Analyse du code

In [ ]:
# Vérifier que le serveur MCP existe
mcp_path = BASE / 'mcp_server.py'
assert mcp_path.exists(), 'mcp_server.py introuvable'

mcp_content = mcp_path.read_text(encoding='utf-8')
print(mcp_content)

# Vérifier les outils exposés
outils_mcp = ['meteo_actuelle', 'meteo_historique', 'previsions_meteo', 
              'recherche_web', 'calculatrice', 'recherche_corpus', 'envoyer_email']

for outil in outils_mcp:
    present = outil in mcp_content
    status = '[OK]' if present else '[KO]'
    print(f'  {status} {outil}')

print('\nPour lancer le serveur MCP : python mcp_server.py')
print('Pour configurer Claude Desktop : ajouter le serveur dans claude_desktop_config.json')
print('>> 4. Serveur MCP : OK')

### Configuration Claude Desktop

Pour connecter notre serveur MCP à Claude Desktop, ajouter dans `claude_desktop_config.json` :

```json
{
  "mcpServers": {
    "rag-catastrophes": {
      "command": "python",
      "args": ["mcp_server.py"],
      "cwd": "chemin/vers/catastrophes-climatiques-rag"
    }
  }
}
```

---

---

## 5. Résultats

In [ ]:
df_outils = pd.DataFrame(results_outils)
csv_path = OUTPUT_DIR / 'NB7_mcp_results.csv'
df_outils.to_csv(csv_path, index=False)
assert csv_path.exists()
print(f'  [OK] {csv_path.name} sauvegardé')
print()
print(df_outils.to_string())

---

## 6. Conclusions

### Quality gate finale

| Constat | Preuve | Décision |
|---|---|---|
| 7 outils fonctionnent en appel direct | Section 3 | Prêts pour MCP |
| Serveur MCP contient les 7 outils | Section 4 | Prêt pour Claude Desktop |
| Configuration Claude Desktop documentée | Section 4 | Démo possible |

---

### Hypothèse : [validée / invalidée]

[À compléter — les outils directs et MCP produisent-ils les mêmes résultats ?]

---

### Limites

- Le test MCP complet nécessite de lancer le serveur en parallèle
- L'email nécessite la configuration Gmail (mot de passe d'application)
- Claude Desktop doit être installé pour la démo MCP

---

### Axes d'amélioration

- Test de charge (10 requêtes consécutives)
- Comparatif latence MCP vs appel direct
- Ajout d'authentification sur le serveur MCP

In [ ]:
elapsed = round(time.time() - notebook_start_time, 2)
print(f'Temps total du notebook : {elapsed}s')
print('>> NOTEBOOK 7 TERMINÉ')